# Exploratory Data Analysis

## Main findings and summary

- **Training data:** The analysis uses 21,900 prompts from the existing training partition, covering 85 task categories and 11 models. The final membership checks confirm that the analyzed prompt IDs exactly match the saved training IDs.

- **Task imbalance:** HellaSwag contributes 5,997 prompts and grade-school-math contributes 4,492. Together, they account for 47.89% of training prompts, giving these tasks substantial influence over prompt-weighted averages.

- **Score distributions:** There are 13 distinct recorded score values. The five levels 0, 0.25, 0.5, 0.75, and 1 account for approximately 99.86% of prompt–model scores. Fractional scores occur in six task categories, with grade-school-math contributing most of them. Mean recorded score should therefore be interpreted as average benchmark quality, rather than automatically as binary accuracy.

- **Model-levels:** GPT-4 has the highest mean recorded score, approximately 0.7800, and the highest average historical cost, approximately $3.298 per 1,000 responses. Mistral 7B has the lowest average historical cost, approximately $0.0456 per 1,000 responses, with a mean score of approximately 0.3072.

- **Multiple cost peaks:** Costs are positive and right-skewed. The pooled log-cost distributions have several peaks, which largely collapse after subtracting each task’s mean log cost.

- **Prompt length:** Prompt length in characters has a strong positive association with cost: overall Spearman correlations range from 0.830 to 0.892 across models. Positive associations persist within the three largest tasks. 

- **Model complementarity:** Other models sometimes outperform GPT-4 on individual prompts despite having lower overall mean scores. Claude-instant scores higher than GPT-4 on 8.15% of training prompts. Yi has the largest average positive score improvement over GPT-4, approximately 0.0534 score units averaged across all training prompts, with non-improvements counted as zero.

- **Perfect-information savings:** Selecting the cheapest model whose recorded score matches or exceeds GPT-4’s on each prompt reduces average historical cost from approximately $3.298 to $0.207 per 1,000 responses—a 93.71% saving. Under equal task weighting, the saving is 87.94%. These are hindsight benchmarks requiring knowledge of every candidate’s outcome. 

- **Repeated prompts:** The training partition contains 12 pairs of identical extracted prompt texts, involving 24 rows. Four pairs differ in at least one model score, and nine differ in at least one model cost.

## Scope and limitations

These findings describe the existing training partition before embedding.

# Code

## Load training data set

In [ ]:
from pathlib import Path
import pandas as pd

DATA_LOCATION = Path(
    "1_data_preparation/data/raw/routerbench_0shot.pkl"
)

PROJECT_DIR = None

for folder in [Path.cwd()] + list(Path.cwd().parents):
    if (folder / DATA_LOCATION).is_file():
        PROJECT_DIR = folder
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Run this notebook from within the project "
        "and check that 1_data_preparation/data/raw/routerbench_0shot.pkl exists."
    )

RAW_PATH = PROJECT_DIR / DATA_LOCATION

df = pd.read_pickle(RAW_PATH)

split_dir = RAW_PATH.parent.parent / "processed" / "char_tfidf_50k"
train_questions = pd.read_pickle(split_dir / "train_questions.pkl")
train_ids = train_questions["sample_id"]

# Keep only raw-data rows whose IDs belong to the saved training split.
df = df.loc[df["sample_id"].isin(train_ids)].copy()

# Stop if IDs are missing, repeated, or do not match the raw dataset.
assert len(train_ids) > 0, "The saved training split is empty."
assert train_ids.notna().all() and train_ids.is_unique, "Invalid training IDs."
assert df["sample_id"].is_unique, "Repeated training IDs in the raw dataset."
assert set(df["sample_id"]) == set(train_ids), "Some training IDs are missing."

print(f"Training partition loaded: {len(df):,} prompts.")

## Dataset features

In [ ]:
df.shape        # Return (number of rows, number of columns).

In [ ]:
df.columns      # Show the column names.
# sample_id, prompt, eval_name: Prompt identifier, prompt text, and task category 
# Columns named after 11 models  11 - Each model’s recorded score for that prompt
# Columns ending in model_response - Each model’s generated answer |
# Columns ending in total_cost - response cost for each model 
# oracle_model_to_route_to -  A stored hindsight model-selection label 

In [ ]:
df.head()       # Show the first five rows.

## Types of Tasks

In [ ]:
# count number of task categories
task_counts = df["eval_name"].value_counts()
print("Number of task categories:", len(task_counts))
task_counts.head(5)


In [ ]:
# percentage of task categories
task_summary = pd.DataFrame({
    "prompt_count": task_counts,
    "percentage": 100 * task_counts / len(df)
})

task_summary.head(5).round(2)

In [ ]:
# Vizualize task categories
#%pip install matplotlib
import matplotlib.pyplot as plt

top_tasks = task_summary["percentage"].head(15)

top_tasks.sort_values().plot.barh(figsize=(10, 6))

plt.xlabel("Percentage of all prompts")
plt.ylabel("Task category")
plt.title("The 15 largest task categories in RouterBench")

plt.tight_layout()
plt.show()

## Scores

Summary: Inspect recorded score levels, fractional scores, and model differences using training rows only.

In [ ]:
# scores dataframe
score_columns = df.columns[3:14]
scores = df[score_columns].astype("float64")
scores.dtypes

In [ ]:
# describe scores
scores.describe().T

In [ ]:
# How many scores are fractional?
score_distribution = pd.DataFrame({
    "Score = 0 (%)": scores.eq(0).mean() * 100,
    "0 < Score < 1 (%)": ((scores > 0) & (scores < 1)).mean() * 100,
    "Score = 1 (%)": scores.eq(1).mean() * 100
})

score_distribution.round(2)

In [ ]:
# Vizualize scores across models
score_distribution.plot.barh(
    stacked=True,
    figsize=(12, 7),
    color=["#D55E00", "#E6C75A", "#009E73"]
)

plt.xlabel("Percentage of prompts")
plt.ylabel("Model")
plt.title("Recorded score composition across all 11 models")
plt.xlim(0, 100)

plt.legend(
    title="Score category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
# What task categories give rise to fractional scores? 
fractional = (scores > 0) & (scores < 1)

fractional_by_task = fractional.groupby(df["eval_name"]).mean() * 100

fractional_by_task.loc[
    fractional_by_task.gt(0).any(axis=1)
].round(2)

In [ ]:
#how does each model’s mean score vary across tasks?
task_scores = scores.groupby(df["eval_name"]).mean()

task_scores.insert(0, "prompt_count", task_counts)

task_scores.sort_values(
    "prompt_count", ascending=False
).head(10).round(3)

In [ ]:
heatmap_data = (
    task_scores
    .sort_values("prompt_count", ascending=False)
    .head(10)
    .drop(columns="prompt_count")
)

plt.figure(figsize=(14, 8))

plt.imshow(
    heatmap_data,
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    aspect="auto"
)

plt.xticks(
    range(len(heatmap_data.columns)),
    heatmap_data.columns,
    rotation=60,
    ha="right"
)

plt.yticks(
    range(len(heatmap_data)),
    heatmap_data.index
)

plt.colorbar(label="Mean recorded score")
plt.title("Mean scores across all 11 models: the 10 largest tasks")

plt.tight_layout()
plt.show()

In [ ]:
# are there distinct scores? 
score_levels = scores.stack().value_counts().sort_index()

print("Number of distinct score values:", len(score_levels))

score_levels

# Five levels dominate: 0, .25, .5, .75, 1.

In [ ]:
# Vizualize score levels 
score_percentages = 100 * score_levels / score_levels.sum()

score_percentages.plot.bar(
    figsize=(10, 5),
    color="steelblue"
)

plt.xlabel("Recorded score level")
plt.ylabel("Percentage of all prompt–model scores")
plt.title("Distribution of observed training score levels")

plt.tight_layout()
plt.show()

In [ ]:
score_details = scores.copy()

score_details["task"] = df["eval_name"]

score_details = score_details.melt(
    id_vars="task",
    var_name="model",
    value_name="score"
)

score_details.head()

In [ ]:
# are scores of a task concentrated at a certain level? 
task_level_counts = pd.crosstab(
    score_details["task"],
    score_details["score"]
)

has_fractional = (
    task_level_counts
    .drop(columns=[0.0, 1.0])
    .sum(axis=1) > 0
)

task_level_counts.loc[has_fractional]

In [ ]:
fractional_task_counts = task_level_counts.loc[has_fractional]

task_level_percentages = fractional_task_counts.div(
    fractional_task_counts.sum(axis=1),
    axis=0
) * 100

task_level_percentages.round(2)

In [ ]:
plt.figure(figsize=(12, 5))

plt.imshow(
    task_level_percentages,
    cmap="Blues",
    vmin=0,
    vmax=100,
    aspect="auto"
)

plt.xticks(
    range(len(task_level_percentages.columns)),
    task_level_percentages.columns
)

plt.yticks(
    range(len(task_level_percentages)),
    task_level_percentages.index
)

plt.xlabel("Recorded score level")
plt.ylabel("Task")
plt.title("Score-level distributions within tasks — all 11 models pooled")
plt.colorbar(label="Percentage of scores within the task")

plt.tight_layout()
plt.show()

## Costs

Summary: Inspect training costs, their relationship to scores, and whether log-cost peaks remain after centering within tasks

In [ ]:
#build cost dataframe

cost_columns = scores.columns + "|total_cost"

costs = df[cost_columns].astype("float64")

costs.columns = scores.columns
costs.head()

In [ ]:
# mean costs for models 
costs.describe(percentiles=[0.5, 0.95, 0.99]).T

In [ ]:
# how much additional mean score accompanies additional cost?
model_summary = pd.DataFrame({
    "mean_score": scores.mean(),
    "cost_per_1000_responses": costs.mean() * 1000
})

model_summary.sort_values(
    "cost_per_1000_responses"
).round(4)

In [ ]:
model_summary.plot.scatter(
    x="cost_per_1000_responses",
    y="mean_score",
    figsize=(12, 7),
    s=70
)

plt.xscale("log")

for model, row in model_summary.iterrows():
    label = model.split("/")[-1]
    offset = (6, -14) if model == "claude-v1" else (6, 6)

    plt.annotate(
        label,
        (row["cost_per_1000_responses"], row["mean_score"]),
        xytext=offset,
        textcoords="offset points",
        fontsize=9
    )

plt.xlabel("Historical dollars per 1,000 responses — logarithmic scale")
plt.ylabel("Mean recorded score")
plt.title("Cost–score comparison across all 11 models")
plt.xlim(0.035, 8)
plt.ylim(0.15, 0.85)

plt.tight_layout()
plt.show()

In [ ]:
cost_diagnostics = pd.DataFrame({
    "mean_to_median": costs.mean() / costs.median(),
    "skewness": costs.skew(),
    "coefficient_of_variation": costs.std() / costs.mean()
})

cost_diagnostics.round(3)

In [ ]:
import numpy as np

log_costs = np.log(costs)

log_costs.hist(
    bins=40,
    figsize=(16, 12),
    layout=(4, 3)
)

plt.tight_layout()
plt.show()

In [ ]:
# Do the peaks remain after centering within tasks.
# Compute 100 (1- within-task-variation/total-variation) 

task_mean_log_cost = log_costs.groupby(
    df["eval_name"]
).transform("mean")

within_task_log_cost = log_costs - task_mean_log_cost

task_explained_percentage = 100 * (
    1 - within_task_log_cost.var() / log_costs.var()
)

task_explained_percentage.sort_values(ascending=False).round(2)


In [ ]:
# plot the costs after centering within tasks
within_task_log_cost.hist(
    bins=40,
    figsize=(16, 12),
    layout=(4, 3)
)

plt.tight_layout()
plt.show()
# The separated peaks have largely collapsed.

In [ ]:
from scipy import stats

fig, axes = plt.subplots(4, 3, figsize=(16, 14))

for ax, model in zip(axes.flat, within_task_log_cost.columns):
    stats.probplot(
        within_task_log_cost[model],
        dist="norm",
        plot=ax
    )

    ax.set_title(model.split("/")[-1])

axes[-1, -1].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
largest_deviation = within_task_log_cost.max(axis=1)

extreme_rows = largest_deviation.nlargest(10).index

extreme_prompts = df.loc[
    extreme_rows, ["sample_id", "eval_name"]
].copy()

extreme_prompts["model"] = (
    within_task_log_cost.loc[extreme_rows].idxmax(axis=1)
)

extreme_prompts["log_deviation"] = largest_deviation.loc[extreme_rows]

extreme_prompts["cost_ratio"] = np.exp(
    extreme_prompts["log_deviation"]
)

extreme_prompts

In [ ]:
#compare all 11 models on the most extreme prompt
row_id = extreme_prompts.index[0]

prompt_comparison = pd.DataFrame({
    "score": scores.loc[row_id],
    "cost_dollars": costs.loc[row_id],
    "relative_cost": np.exp(within_task_log_cost.loc[row_id])
})

prompt_comparison.sort_values(
    "relative_cost", ascending=False
)

In [ ]:
# compare response lengths across all 11 models. A longer generated response is one possible explanation for the unusually high cost
response_columns = scores.columns + "|model_response"

responses = df.loc[row_id, response_columns].copy()

responses.index = scores.columns

response_comparison = prompt_comparison.copy()

response_comparison["stored_response_characters"] = responses.str.len()

response_comparison.sort_values(
    "stored_response_characters", ascending=False
)

In [ ]:
mistral_response = responses["mistralai/mistral-7b-chat"]

print("PROMPT:")
print(df.loc[row_id, "prompt"])

print("\nMISTRAL — FIRST 1500 CHARACTERS:")
print(mistral_response[:1500])

print("\nMISTRAL — LAST 500 CHARACTERS:")
print(mistral_response[-500:])

print("\nYI RESPONSE:")
print(responses["zero-one-ai/Yi-34B-Chat"])


# Prompts

In [ ]:
# prompt-length analysis
import ast

prompt_parts = df["prompt"].apply(ast.literal_eval)

prompt_text = prompt_parts.str.join("\n\n").str.strip()

prompt_lengths = prompt_text.str.len()

prompt_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99])

In [ ]:
# Plot the prompt-length distribution
prompt_lengths.plot.hist(
    bins=50,
    figsize=(10, 5),
    color="steelblue",
    edgecolor="white"
)

plt.axvline(
    prompt_lengths.median(),
    color="darkorange",
    linestyle="--",
    label="Median"
)

plt.xlabel("Prompt length in characters")
plt.ylabel("Number of prompts")
plt.title("Distribution of prompt lengths")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Measure how length relates to score and cost
length_relationships = pd.DataFrame({
    "length_vs_score": scores.corrwith(
        prompt_lengths, method="spearman"
    ),
    "length_vs_cost": costs.corrwith(
        prompt_lengths, method="spearman"
    )
})

length_relationships.round(3)

In [ ]:
# does the length - cost relationship persist within tasks?
within_task_results = {}

for task in task_counts.head(3).index:
    mask = df["eval_name"].eq(task)

    within_task_results[task] = costs.loc[mask].corrwith(
        prompt_lengths.loc[mask],
        method="spearman"
    )

within_task_length_cost = pd.DataFrame(within_task_results)

within_task_length_cost.round(3)

In [ ]:
# does the length - score relationship persist within tasks?
within_task_score_results = {}

for task in task_counts.head(3).index:
    mask = df["eval_name"].eq(task)

    within_task_score_results[task] = scores.loc[mask].corrwith(
        prompt_lengths.loc[mask],
        method="spearman"
    )

within_task_length_score = pd.DataFrame(
    within_task_score_results
)

within_task_length_score.round(3)

## Comparing models

Summary: Choose the highest-mean model on training data as an exploratory reference. Compare paired scores and costs, then calculate perfect-information savings

In [ ]:
# Even though GPT-4 has the highest overall mean score, how often does another model score higher on the same prompt?
reference_model = scores.mean().idxmax()

reference_scores = scores[reference_model]

paired_comparison = pd.DataFrame({
    "higher_score_percent":
        scores.gt(reference_scores, axis=0).mean() * 100,

    "equal_score_percent":
        scores.eq(reference_scores, axis=0).mean() * 100,

    "lower_score_percent":
        scores.lt(reference_scores, axis=0).mean() * 100
})

print("Exploratory reference:", reference_model)

paired_comparison.sort_values(
    "higher_score_percent", ascending=False
).round(2)
# GPT 4 doesn’t score highest on every prompt

In [ ]:
# measure the magnitude of gains and losses by choosing a different model than GPT 4
score_differences = scores.sub(reference_scores, axis=0)

gain_and_loss = pd.DataFrame({
    "mean_gain": score_differences.clip(lower=0).mean(),

    "mean_loss": (-score_differences).clip(lower=0).mean(),

    "net_difference": score_differences.mean()
})

gain_and_loss.sort_values(
    "mean_gain", ascending=False
).round(4)

In [ ]:
# how often is another model cheaper without lowering the recorded score?
reference_costs = costs[reference_model]

no_lower_score = scores.ge(reference_scores, axis=0)

lower_cost = costs.lt(reference_costs, axis=0)

cheaper_without_score_loss = no_lower_score & lower_cost

(
    cheaper_without_score_loss.mean() * 100
).sort_values(ascending=False).round(2)


In [ ]:
# If we knew every outcome beforehand, what would be the cheapest model on each prompt whose score was at least GPT-4’s?
eligible_costs = costs.where(no_lower_score)

hindsight_cost = eligible_costs.min(axis=1)

savings_fraction = (
    1 - hindsight_cost.mean() / reference_costs.mean()
)

print(
    "GPT-4 dollars per 1,000 responses:",
    reference_costs.mean() * 1000
)

print(
    "Hindsight dollars per 1,000 responses:",
    hindsight_cost.mean() * 1000
)

print("Aggregate cost savings (%):", savings_fraction * 100)


In [ ]:
# where do those savings come from?
hindsight_model = eligible_costs.idxmin(axis=1)

routing_counts = hindsight_model.value_counts().reindex(
    scores.columns, fill_value=0
)

spending_by_model = hindsight_cost.groupby(
    hindsight_model
).sum().reindex(scores.columns, fill_value=0)

routing_summary = pd.DataFrame({
    "prompt_count": routing_counts,
    "prompt_percent": 100 * routing_counts / len(df),
    "spending_percent": 100 * spending_by_model / hindsight_cost.sum()
})

routing_summary.sort_values(
    "prompt_percent", ascending=False
).round(2)

In [ ]:
# are the savings concentrated on prompts where GPT-4 scored zero?
benchmark_details = pd.DataFrame({
    "reference_score": reference_scores,
    "reference_cost": reference_costs,
    "hindsight_cost": hindsight_cost
})

by_reference_score = benchmark_details.groupby(
    "reference_score"
).agg(
    prompt_count=("reference_cost", "size"),
    reference_mean_cost=("reference_cost", "mean"),
    hindsight_mean_cost=("hindsight_cost", "mean")
)

by_reference_score["savings_percent"] = 100 * (
    1
    - by_reference_score["hindsight_mean_cost"]
    / by_reference_score["reference_mean_cost"]
)

by_reference_score.round(6)

In [ ]:
# which tasks account for the spending that remains?

task_benchmark = benchmark_details.assign(
    task=df["eval_name"]
).groupby("task").agg(
    prompt_count=("reference_cost", "size"),
    reference_spend=("reference_cost", "sum"),
    hindsight_spend=("hindsight_cost", "sum")
)

task_benchmark["savings_percent"] = 100 * (
    1
    - task_benchmark["hindsight_spend"]
    / task_benchmark["reference_spend"]
)

task_benchmark["hindsight_spend_share"] = 100 * (
    task_benchmark["hindsight_spend"]
    / task_benchmark["hindsight_spend"].sum()
)

task_benchmark.sort_values(
    "hindsight_spend", ascending=False
).head(10).round(4)

In [ ]:
# rank tasks by cost per response instead of total spending
task_benchmark["reference_cost_per_1000"] = (
    1000
    * task_benchmark["reference_spend"]
    / task_benchmark["prompt_count"]
)

task_benchmark["hindsight_cost_per_1000"] = (
    1000
    * task_benchmark["hindsight_spend"]
    / task_benchmark["prompt_count"]
)

task_benchmark.sort_values(
    "hindsight_cost_per_1000", ascending=False
)[[
    "prompt_count",
    "reference_cost_per_1000",
    "hindsight_cost_per_1000",
    "savings_percent"
]].head(10).round(3)

In [ ]:
# check how task weighting affects the overall savings. What if we instead give every task equal weight?
weighting_comparison = pd.DataFrame({
    "reference_cost_per_1000": [
        reference_costs.mean() * 1000,
        task_benchmark["reference_cost_per_1000"].mean()
    ],
    "hindsight_cost_per_1000": [
        hindsight_cost.mean() * 1000,
        task_benchmark["hindsight_cost_per_1000"].mean()
    ]
}, index=[
    "Equal weight per prompt",
    "Equal weight per task"
])

weighting_comparison["savings_percent"] = 100 * (
    1
    - weighting_comparison["hindsight_cost_per_1000"]
    / weighting_comparison["reference_cost_per_1000"]
)

weighting_comparison.round(3)

## Some other checks

In [ ]:
# check repeated prompts
duplicate_check = pd.Series({
    "Duplicate IDs beyond the first occurrence":
        df["sample_id"].duplicated().sum(),

    "Duplicate prompt texts beyond the first occurrence":
        prompt_text.duplicated().sum(),

    "Rows belonging to repeated-text groups":
        prompt_text.duplicated(keep=False).sum()
})

duplicate_check

In [ ]:
# those paired records have identical scores and costs?

duplicate_mask = prompt_text.duplicated(keep=False)

duplicate_groups = prompt_text.loc[duplicate_mask]

score_variation = (
    scores.loc[duplicate_mask]
    .groupby(duplicate_groups)
    .nunique()
)

cost_variation = (
    costs.loc[duplicate_mask]
    .groupby(duplicate_groups)
    .nunique()
)

print(
    "Groups with at least one differing model score:",
    score_variation.gt(1).any(axis=1).sum()
)

print(
    "Groups with at least one differing model cost:",
    cost_variation.gt(1).any(axis=1).sum()
)

In [ ]:
# check the overlap and task labels
duplicate_audit = pd.DataFrame({
    "score_differs": score_variation.gt(1).any(axis=1),

    "cost_differs": cost_variation.gt(1).any(axis=1),

    "task_count": (
        df.loc[duplicate_mask]
        .groupby(duplicate_groups)["eval_name"]
        .nunique()
    )
})

duplicate_audit.groupby(
    ["score_differs", "cost_differs", "task_count"]
).size()

In [ ]:
# Verify that this EDA still contains exactly the saved training rows.
assert df["sample_id"].is_unique
assert set(df["sample_id"]) == set(train_ids)

print("Rows in saved training partition:", len(train_ids))
print("Rows used in this EDA:", len(df))
print("Training IDs match exactly. Existing splits were not changed.")